# 三种RAG模式复现查询菜谱系统

三种RAG
1. 基础rag ： 问，检索知识，把结果给llm，生成回答
2. agentic rag ： 问，agent判断是否调用tools，调用tools，回答 （检索变成Agent工具）
3. corrective rag ： 问，检索知识库，判断是否相关，相关回答，不相关则不回答

In [1]:
%pip install langchain langchain-openai langchain-community langchain-oceanbase
%pip install langchain-text-splitters pypdf pymysql python-dotenv
%pip install sentence-transformers langchain-huggingface pyobvector

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
     ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
     ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
     ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
     -------- ------------------------------- 0.5/2.5 MB 644.1 kB/s eta 0:00:04
     -------- ------------------------------- 0.5/2.5 MB 644.1 kB/s eta 0:00:04
     -------- ------------------------------- 0.5/2.5 MB 644.1 kB/s eta 0:00:04
     ------------ --------------------------- 0.8/2.5 MB 524.3 kB/s eta 0:00:04
     ---------------- ----------------------- 1.0/2.5 MB 653.7 kB/s eta 0:00:03
     ---------------- ----------------------- 1.0/2.5 MB 653.7 kB/s eta 0:00:03
     ---------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-experimental 0.3.4 requires langchain-community<0.4.0,>=0.3.0, but you have langchain-community 0.4.1 which is incompatible.
langchain-experimental 0.3.4 requires langchain-core<0.4.0,>=0.3.28, but you have langchain-core 1.3.3 which is incompatible.


Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://mirrors.aliyun.com/pypi/simple/

  Attempting uninstall: packaging

    Found existing installation: packaging 26.0

    Uninstalling packaging-26.0:

      Successfully uninstalled packaging-26.0

   ---------------------------------------- 0/2 [packaging]
  Attempting uninstall: langchain-core
   ---------------------------------------- 0/2 [packaging]
   -------------------- ------------------- 1/2 [langchain-core]
    Found existing installation: langchain-core 1.3.3
   -------------------- ------------------- 1/2 [langchain-core]
   -------------------- ------------------- 1/2 [langchain-core]
   -------------------- ------------------- 1/2 [langchain-core]
    Uninstalling langchain-core-1.3.3:
   -------------------- ------------------- 1/2 [langchain-core]
   -------------------- ------------------- 1/2 [langchain-core]
      

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.2.18 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.3.86 which is incompatible.
langchain-anthropic 1.4.0 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.4 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.86 which is incompatible.
langchain-community 0.4.1 requires langchain-core<2.0.0,>=1.0.1, but you have langchain-core 0.3.86 which is incompatible.
langchain-deepseek 0.1.4 requires langchain-openai<1.0.0,>=0.3.28, but you have langchain-openai 1.2.1 which is incompatible.
langchain-experimental 0.3.4 requires langchain-community<0.4.0,>=0.3.0, but you have langchain-community 0.4.1 which is incompatible.
langchain-mcp-adapters 0.2.2 requires langchain-core<2.0.0,>=1.0.0, but you have 

# 启动OceanBase数据库

In [ ]:
# 命令启动 ： docker compose -f docker-compose.oceanbase.yml up -d
# 检查docker状态 ： docker ps

# 配置.emv文件

配置文件如下
```.env
# 阿里百炼
BASE_URL=
API_KEY=

AIHUBMIX_API_KEY = 
AIHUBMIX_BASE_URL = 
AIHUBMIX_MODEL = "gpt-4.1-free"


#FAISS 向量数据库配置
FAISS_DB_DIR=./faiss_db

OB_HOST=127.0.0.1
OB_PORT=2881
OB_USER=root@test
OB_PASSWORD=
OB_DATABASE=test
```

# 基础RAG

In [ ]:
from dotenv import load_dotenv
import os 
import pymysql

load_dotenv() # 加载环境变量

# 从环境变量中获取信息
args = {
    "ob_host": os.getenv("OB_HOST"),
    "ob_port": os.getenv("OB_PORT"),
    "ob_user": os.getenv("OB_USER"),
    "ob_password": os.getenv("OB_PASSWORD"),
    "ob_database": os.getenv("OB_DATABASE")
}

conn = pymysql.connect(
    host = args["ob_host"],
    port = int(args["ob_port"]),
    user = args["ob_user"],
    password = args["ob_password"],
    database = args["ob_database"],
    charset = "utf8mb4"
m    connect_timeout = 10
)

# 测试链接
with conn.cursor() as cursor:
    cursor.execute("SELECT VERSION()")
    version = cursor.fetchone()
    print("OceanBase Version:", version[0])
    conn.close()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4194149692.py, line 21)